# ROBERT Context Extractor

This notebook reads the output files from a completed ROBERT run and produces a single structured file called `run_context.json`.

That JSON file is the foundation for everything that comes next: diagnosing why a score is high or low, and producing plain-language explanations for chemists.

**What this notebook does:**
- Points to an archived ROBERT run folder (created by `robert_run_wrapper.ipynb`)
- Reads `PREDICT_data.dat` and `VERIFY_data.dat` (and optionally `CURATE_data.dat`)
- Extracts numbers and flags from those text files in a reliable, null-safe way
- Writes a `run_context.json` to the same run folder

**What this notebook does NOT do:**
- Re-run ROBERT
- Diagnose the score (that is the next notebook)
- Call any AI or external service

**Audience note:** The cells below include plain-language explanations before each code block. You do not need to read or edit the code to use this notebook — only Cell 3 needs to be updated for each new run.

## Cell 2 Guide: Point to a Run Folder

This is the only cell you need to edit for each new run.

Set `RUN_FOLDER` to the path of the archived run you want to inspect.
You will find archived run folders inside `agent/run_archive/` — each one is named
with a timestamp and the dataset name, for example:

```
agent/run_archive/20260513_160812__Hvapor/
```

You can either:
- paste the full path (e.g. `"/Users/.../agent/run_archive/20260513_160812__Hvapor"`), or
- leave `RUN_FOLDER = None` to auto-select the most recent run in `run_archive/`.

The extracted `run_context.json` will be written into that same run folder.

In [79]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import shutil

# -----------------------------------------------------------------------
# EDIT THIS: path to the run folder you want to extract context from.
# Set to None to auto-select the most recently created run.
# -----------------------------------------------------------------------
RUN_FOLDER = None

# -----------------------------------------------------------------------
# Auto-resolve project root (same logic as the wrapper notebook)
# -----------------------------------------------------------------------
def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "AGENTS.md").exists() and (candidate / "robert").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not infer project root. "
        "Expected a folder containing AGENTS.md and robert/."
    )

PROJECT_ROOT = resolve_project_root()
RUNS_ROOT = PROJECT_ROOT / "agent" / "run_archive"

if RUN_FOLDER is None:
    # Auto-select the most recently created run folder
    all_runs = sorted(
        [p for p in RUNS_ROOT.iterdir() if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not all_runs:
        raise FileNotFoundError(
            f"No run folders found in {RUNS_ROOT}. "
            "Run agent/robert_run_wrapper.ipynb first."
        )
    RUN_FOLDER = all_runs[0]
    print(f"Auto-selected most recent run: {RUN_FOLDER.name}")
else:
    RUN_FOLDER = Path(RUN_FOLDER).resolve()

if not RUN_FOLDER.exists():
    raise FileNotFoundError(f"Run folder not found: {RUN_FOLDER}")

OUTPUTS_DIR = RUN_FOLDER / "outputs"

print(f"Project root : {PROJECT_ROOT}")
print(f"Run folder   : {RUN_FOLDER}")
print(f"Outputs dir  : {OUTPUTS_DIR}")

Auto-selected most recent run: 20260521_062044__AQME-ROBERT_interpret_CO2
Project root : /Users/cjcscha/ROBERT/helper_rob/robert
Run folder   : /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260521_062044__AQME-ROBERT_interpret_CO2
Outputs dir  : /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260521_062044__AQME-ROBERT_interpret_CO2/outputs


## Cell 4 Guide: Check Which Output Files Are Present

ROBERT produces four output folders: `CURATE`, `GENERATE`, `VERIFY`, and `PREDICT`.
If a run stopped early, some folders may be missing.

This cell checks which `.dat` files exist and sets availability flags used throughout the rest of the notebook.
It also reads the `run_manifest.json` if it exists, which records the exact command used.

In [80]:
# Locate the key .dat files inside the outputs/ folder
PREDICT_DAT  = OUTPUTS_DIR / "PREDICT" / "PREDICT_data.dat"
VERIFY_DAT   = OUTPUTS_DIR / "VERIFY"  / "VERIFY_data.dat"
CURATE_DAT   = OUTPUTS_DIR / "CURATE"  / "CURATE_data.dat"
GENERATE_DAT = OUTPUTS_DIR / "GENERATE" / "GENERATE_data.dat"
MANIFEST_JSON = RUN_FOLDER / "run_manifest.json"

# Read each file into a list of lines, or None if the file is missing.
# This approach means downstream cells never crash on a missing file.
def read_dat(path: Path):
    """Return list of lines from a .dat file, or None if missing/unreadable."""
    if not path.exists():
        return None
    try:
        return path.read_text(encoding="utf-8").splitlines()
    except Exception as e:
        print(f"WARNING: could not read {path}: {e}")
        return None


def dedupe_paths(paths):
    seen = set()
    unique = []
    for p in paths:
        key = str(p.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(p.resolve())
    return unique


def sanitize_tag(text: str) -> str:
    cleaned = []
    for ch in text:
        if ch.isalnum() or ch in ["-", "_"]:
            cleaned.append(ch)
        else:
            cleaned.append("_")
    return "".join(cleaned).strip("_") or "dataset"


def discover_report_pdfs(project_root: Path, run_folder: Path, outputs_dir: Path, dataset_csv: str | None = None):
    """Find report PDFs from authoritative sources and copy into archive with deterministic names.

    Output naming format:
    <dataset_stem>__<run_timestamp>__ROBERT_report[__N].pdf
    """
    candidates = []

    # Primary source: standard ROBERT report in project root
    root_pdf = project_root / "ROBERT_report.pdf"
    if root_pdf.exists():
        candidates.append(root_pdf)

    # Secondary source: module outputs (if PDFs are emitted there in some workflows)
    if outputs_dir.exists():
        candidates.extend([p for p in outputs_dir.glob("**/*.pdf") if p.is_file()])

    candidates = dedupe_paths([p for p in candidates if p.exists() and p.is_file()])

    report_assets_dir = run_folder / "report_assets"
    report_assets_dir.mkdir(parents=True, exist_ok=True)

    run_name = run_folder.name
    if "__" in run_name:
        run_timestamp, fallback_dataset = run_name.split("__", 1)
    else:
        run_timestamp, fallback_dataset = datetime.now().strftime("%Y%m%d_%H%M%S"), run_name

    dataset_tag = fallback_dataset
    if dataset_csv:
        try:
            dataset_tag = Path(dataset_csv).stem
        except Exception:
            dataset_tag = fallback_dataset
    dataset_tag = sanitize_tag(dataset_tag)

    # Keep report_assets clean and deterministic per run.
    for old in report_assets_dir.glob("*.pdf"):
        try:
            old.unlink()
        except Exception:
            pass

    archived = []
    warnings = []
    for idx, src in enumerate(candidates, start=1):
        base = f"{dataset_tag}__{run_timestamp}__ROBERT_report"
        suffix_part = "" if idx == 1 else f"__{idx}"
        dest = report_assets_dir / f"{base}{suffix_part}.pdf"

        try:
            shutil.copy2(src, dest)
        except Exception as e:
            warnings.append(f"Could not copy report PDF {src} -> {dest}: {e}")
            continue

        archived.append({
            "name": dest.name,
            "path": str(dest),
            "source_path": str(src),
            "dataset_tag": dataset_tag,
            "run_timestamp": run_timestamp,
        })

    return archived, warnings


predict_lines  = read_dat(PREDICT_DAT)
verify_lines   = read_dat(VERIFY_DAT)
curate_lines   = read_dat(CURATE_DAT)
generate_lines = read_dat(GENERATE_DAT)

# Set availability flags for the run_context.json
avail_predict  = predict_lines  is not None
avail_verify   = verify_lines   is not None
avail_curate   = curate_lines   is not None
avail_generate = generate_lines is not None

# Read run manifest for provenance
manifest_data = None
if MANIFEST_JSON.exists():
    try:
        manifest_data = json.loads(MANIFEST_JSON.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"WARNING: could not read manifest: {e}")

dataset_csv_from_manifest = manifest_data.get("dataset_csv") if manifest_data else None

# Discover and archive report PDFs so UI can view them from run_archive
report_pdfs, report_pdf_warnings = discover_report_pdfs(
    PROJECT_ROOT,
    RUN_FOLDER,
    OUTPUTS_DIR,
    dataset_csv=dataset_csv_from_manifest,
)

print(f"PREDICT_data.dat  : {'FOUND' if avail_predict  else 'MISSING'}")
print(f"VERIFY_data.dat   : {'FOUND' if avail_verify   else 'MISSING'}")
print(f"CURATE_data.dat   : {'FOUND' if avail_curate   else 'MISSING'}")
print(f"GENERATE_data.dat : {'FOUND' if avail_generate else 'MISSING'}")
print(f"run_manifest.json : {'FOUND' if manifest_data is not None else 'MISSING'}")
print(f"Report PDFs linked: {len(report_pdfs)}")
if report_pdfs:
    for item in report_pdfs:
        print(f"  - {item['path']}")
if report_pdf_warnings:
    print("Report PDF warnings:")
    for w in report_pdf_warnings:
        print(f"  - {w}")

PREDICT_data.dat  : FOUND
VERIFY_data.dat   : FOUND
CURATE_data.dat   : FOUND
GENERATE_data.dat : FOUND
run_manifest.json : FOUND
Report PDFs linked: 1
  - /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260521_062044__AQME-ROBERT_interpret_CO2/report_assets/AQME-ROBERT_interpret_CO2__20260521_062044__ROBERT_report.pdf


## Cell 6 Guide: Parser Helper

This cell defines a small helper function used throughout the notebook.

It contains a shared utility (`safe_float`) and the function that splits a `.dat` file
into its two logical sections: one for the model trained on all variables (`No PFI`)
and one for the model trained on only the most important variables (`PFI` — Permutation
Feature Importance filter).

Most ROBERT runs produce both variants. The extractor handles each one independently
so they can be compared.

In [81]:
def safe_float(value_str, warnings_list=None, label=""):
    """
    Convert a string to float. Returns None if conversion fails.
    Appends a descriptive message to warnings_list if provided.
    """
    if value_str is None:
        return None
    try:
        return float(value_str.strip())
    except (ValueError, AttributeError):
        msg = f"Could not convert to float: {label!r} = {value_str!r}"
        if warnings_list is not None:
            warnings_list.append(msg)
        return None


def safe_int(value_str, warnings_list=None, label=""):
    """
    Convert a string to int. Returns None if conversion fails.
    """
    if value_str is None:
        return None
    try:
        return int(value_str.strip())
    except (ValueError, AttributeError):
        msg = f"Could not convert to int: {label!r} = {value_str!r}"
        if warnings_list is not None:
            warnings_list.append(msg)
        return None


def split_into_blocks(lines, no_pfi_marker, pfi_marker):
    """
    Split a list of .dat file lines into two blocks:
    - 'no_pfi': lines in the No PFI section
    - 'pfi': lines in the PFI section

    Returns a dict: {'no_pfi': [...], 'pfi': [...]}
    Each value is an empty list if the section was not found.
    """
    blocks = {"no_pfi": [], "pfi": []}
    if lines is None:
        return blocks

    current = None
    for line in lines:
        if no_pfi_marker in line:
            current = "no_pfi"
        elif pfi_marker in line:
            current = "pfi"
        if current is not None:
            blocks[current].append(line)

    return blocks


# The section headers used in PREDICT_data.dat and VERIFY_data.dat
NO_PFI_MARKER = "Starting model with all variables (No PFI)"
PFI_MARKER    = "Starting model with PFI filter"

print("Helper functions defined.")

Helper functions defined.


## Cell 8 Guide: Parse PREDICT_data.dat

This is the most information-rich file. For each model variant (No PFI and PFI),
ROBERT records:

- which descriptors (molecular features) were used,
- how many training and test compounds were used,
- cross-validation (CV) and test-set performance (R², MAE, RMSE for regression;
  MCC for classification),
- how uniform the data distribution was across the prediction range,
- which compounds were flagged as outliers.

The code extracts each of these values using pattern matching on the text lines.
If a line is missing or formatted differently (for example in a classification run),
the corresponding field is set to `None` rather than crashing.

In [82]:
def parse_predict_block(block_lines, warnings):
    """
    Extract performance metrics from one section (No PFI or PFI) of PREDICT_data.dat.

    This parser is evidence-first: it extracts values directly from ROBERT text output.
    """
    result = {
        "cv_type"             : None,
        "points_descp_ratio"  : None,

        # Regression metrics
        "r2_cv"               : None,
        "r2_test"             : None,
        "rmse_cv"             : None,
        "rmse_test"           : None,
        "mae_cv"              : None,
        "mae_test"            : None,

        # Classification metrics (when present)
        "mcc_cv"              : None,
        "mcc_test"            : None,
        "f1_cv"               : None,
        "f1_test"             : None,
        "accuracy_cv"         : None,
        "accuracy_test"       : None,

        "avg_sd_test"         : None,
        "y_min"               : None,
        "y_max"               : None,
        "y_range"             : None,
        "n_train"             : None,
        "n_test"              : None,
        "n_descriptors"       : None,
        "descriptors"         : None,
        "model"               : None,
        "target"              : None,
        "kfold"               : None,
        "cv_repeats"          : None,

        # Outlier summaries and explicit lists
        "train_outlier_count" : None,
        "train_outlier_pct"   : None,
        "test_outlier_count"  : None,
        "test_outlier_pct"    : None,
        "train_outliers"      : [],
        "test_outliers"       : [],

        # y-distribution
        "quartile_counts"     : None,
        "y_distribution_warning": None,

        # Feature importance text from PFI plot section in PREDICT_data.dat
        "feature_importance_summary": [],

        # Paths reported by ROBERT for generated artifacts
        "artifacts"           : {
            "results_plot": None,
            "cv_variability_plot": None,
            "shap_plot": None,
            "pfi_plot": None,
            "pearson_heatmap": None,
            "outliers_plot": None,
            "y_distribution_plot": None,
            "predictions_csv": None,
        },

        # Warning lines as printed by ROBERT
        "warnings"            : [],
    }

    if not block_lines:
        return result

    full_text = "\n".join(block_lines)

    # ----------------------------------------------------------------
    # Model and target metadata
    # ----------------------------------------------------------------
    m = re.search(r"-\s+Model:\s+(\S+)", full_text)
    if m:
        result["model"] = m.group(1).strip()

    m = re.search(r"-\s+Target value:\s+(.+)", full_text)
    if m:
        result["target"] = m.group(1).strip()

    m = re.search(r"-\s+k-fold CV:\s+(\d+)", full_text)
    if m:
        result["kfold"] = safe_int(m.group(1), warnings, "kfold")

    m = re.search(r"-\s+Repetitions CV:\s+(\d+)", full_text)
    if m:
        result["cv_repeats"] = safe_int(m.group(1), warnings, "cv_repeats")

    # ----------------------------------------------------------------
    # Descriptor list
    # ----------------------------------------------------------------
    m = re.search(r"-\s+Descriptors:\s+(\[.+?\])", full_text)
    if m:
        try:
            result["descriptors"] = json.loads(m.group(1).replace("'", '"'))
        except Exception:
            result["descriptors"] = m.group(1)

    # ----------------------------------------------------------------
    # Training and test point counts
    # ----------------------------------------------------------------
    m = re.search(r"-\s+Training points:\s+(\d+)", full_text)
    if m:
        result["n_train"] = safe_int(m.group(1), warnings, "n_train")

    m = re.search(r"-\s+Test points:\s+(\d+)", full_text)
    if m:
        result["n_test"] = safe_int(m.group(1), warnings, "n_test")

    m = re.search(r"Proportion \(train\+valid\.\) points:descriptors\s*=\s*([\d]+:[\d]+)", full_text)
    if m:
        result["points_descp_ratio"] = m.group(1)

    m = re.search(r"-\s+Number of descriptors\s*=\s*(\d+)", full_text)
    if m:
        result["n_descriptors"] = safe_int(m.group(1), warnings, "n_descriptors")

    # ----------------------------------------------------------------
    # CV and test metrics
    # ----------------------------------------------------------------
    cv_pattern_reg = r"(\d+)x\s+(\d+)-fold CV\s*:\s+R2\s*=\s*([\d.eE+\-]+),\s*MAE\s*=\s*([\d.eE+\-]+),\s*RMSE\s*=\s*([\d.eE+\-]+)"
    cv_pattern_clas = r"(\d+)x\s+(\d+)-fold CV\s*:\s+Accur\.\s*=\s*([\d.eE+\-]+),\s*F1 score\s*=\s*([\d.eE+\-]+),\s*MCC\s*=\s*([\d.eE+\-]+)"
    cv_pattern_clas_alt = r"(\d+)x\s+(\d+)-fold CV\s*:\s+MCC\s*=\s*([\d.eE+\-]+)(?:,\s*F1\s*=\s*([\d.eE+\-]+))?(?:,\s*(?:ACC|Accuracy)\s*=\s*([\d.eE+\-]+))?"

    m = re.search(cv_pattern_reg, full_text)
    if m:
        result["cv_type"] = f"{m.group(1)}x {m.group(2)}-fold CV"
        result["r2_cv"] = safe_float(m.group(3), warnings, "r2_cv")
        result["mae_cv"] = safe_float(m.group(4), warnings, "mae_cv")
        result["rmse_cv"] = safe_float(m.group(5), warnings, "rmse_cv")
    else:
        m = re.search(cv_pattern_clas, full_text)
        if m:
            result["cv_type"] = f"{m.group(1)}x {m.group(2)}-fold CV"
            result["accuracy_cv"] = safe_float(m.group(3), warnings, "accuracy_cv")
            result["f1_cv"] = safe_float(m.group(4), warnings, "f1_cv")
            result["mcc_cv"] = safe_float(m.group(5), warnings, "mcc_cv")
            result["r2_cv"] = result["mcc_cv"]
        else:
            m = re.search(cv_pattern_clas_alt, full_text)
            if m:
                result["cv_type"] = f"{m.group(1)}x {m.group(2)}-fold CV"
                result["mcc_cv"] = safe_float(m.group(3), warnings, "mcc_cv")
                result["r2_cv"] = result["mcc_cv"]
                result["f1_cv"] = safe_float(m.group(4), warnings, "f1_cv") if m.group(4) is not None else None
                result["accuracy_cv"] = safe_float(m.group(5), warnings, "accuracy_cv") if m.group(5) is not None else None

    test_pattern_reg = r"-\s+Test\s*:\s+R2\s*=\s*([\d.eE+\-]+),\s*MAE\s*=\s*([\d.eE+\-]+),\s*RMSE\s*=\s*([\d.eE+\-]+)"
    test_pattern_clas = r"-\s+Test\s*:\s+Accur\.\s*=\s*([\d.eE+\-]+),\s*F1 score\s*=\s*([\d.eE+\-]+),\s*MCC\s*=\s*([\d.eE+\-]+)"
    test_pattern_clas_alt = r"-\s+Test\s*:\s+MCC\s*=\s*([\d.eE+\-]+)(?:,\s*F1\s*=\s*([\d.eE+\-]+))?(?:,\s*(?:ACC|Accuracy)\s*=\s*([\d.eE+\-]+))?"

    m = re.search(test_pattern_reg, full_text)
    if m:
        result["r2_test"] = safe_float(m.group(1), warnings, "r2_test")
        result["mae_test"] = safe_float(m.group(2), warnings, "mae_test")
        result["rmse_test"] = safe_float(m.group(3), warnings, "rmse_test")
    else:
        m = re.search(test_pattern_clas, full_text)
        if m:
            result["accuracy_test"] = safe_float(m.group(1), warnings, "accuracy_test")
            result["f1_test"] = safe_float(m.group(2), warnings, "f1_test")
            result["mcc_test"] = safe_float(m.group(3), warnings, "mcc_test")
            result["r2_test"] = result["mcc_test"]
        else:
            m = re.search(test_pattern_clas_alt, full_text)
            if m:
                result["mcc_test"] = safe_float(m.group(1), warnings, "mcc_test")
                result["r2_test"] = result["mcc_test"]
                result["f1_test"] = safe_float(m.group(2), warnings, "f1_test") if m.group(2) is not None else None
                result["accuracy_test"] = safe_float(m.group(3), warnings, "accuracy_test") if m.group(3) is not None else None

    # ----------------------------------------------------------------
    # Average SD in test set and y-range
    # ----------------------------------------------------------------
    m = re.search(r"Average SD in test set\s*=\s*([\d.eE+\-]+)", full_text)
    if m:
        result["avg_sd_test"] = safe_float(m.group(1), warnings, "avg_sd_test")

    m = re.search(
        r"y range of dataset \(train\+valid\.\)\s*=\s*([\d.eE+\-]+) to ([\d.eE+\-]+),\s*total ([\d.eE+\-]+)",
        full_text,
    )
    if m:
        result["y_min"] = safe_float(m.group(1), warnings, "y_min")
        result["y_max"] = safe_float(m.group(2), warnings, "y_max")
        result["y_range"] = safe_float(m.group(3), warnings, "y_range")

    # ----------------------------------------------------------------
    # Extract artifact paths as reported by ROBERT
    # ----------------------------------------------------------------
    for line in block_lines:
        s = line.strip()
        if "Predicted results of starting dataset:" in s:
            result["artifacts"]["predictions_csv"] = s.split(":", 1)[-1].strip()
        elif "Graph in:" in s and result["artifacts"]["results_plot"] is None:
            result["artifacts"]["results_plot"] = s.split(":", 1)[-1].strip()
        elif "Graph in:" in s and result["artifacts"]["cv_variability_plot"] is None:
            result["artifacts"]["cv_variability_plot"] = s.split(":", 1)[-1].strip()
        elif "SHAP plot saved in" in s:
            result["artifacts"]["shap_plot"] = s.split("saved in", 1)[-1].strip()
        elif "PFI plot saved in" in s:
            result["artifacts"]["pfi_plot"] = s.split("saved in", 1)[-1].strip()
        elif "Pearson heatmap" in s and "stored in" in s:
            result["artifacts"]["pearson_heatmap"] = s.split("stored in", 1)[-1].strip().rstrip(".")
        elif "Outliers plot saved in" in s:
            result["artifacts"]["outliers_plot"] = s.split("saved in", 1)[-1].strip()
        elif "y-values distribution plot saved in" in s:
            result["artifacts"]["y_distribution_plot"] = s.split("saved in", 1)[-1].strip()

    # ----------------------------------------------------------------
    # Outlier counts and explicit outlier lists
    # ----------------------------------------------------------------
    for idx, line in enumerate(block_lines):
        s = line.strip()

        m = re.search(r"Train:\s+(\d+) outliers out of (\d+) datapoints \(([\d.]+)%\)", s)
        if m:
            result["train_outlier_count"] = safe_int(m.group(1), warnings, "train_outlier_count")
            result["train_outlier_pct"] = safe_float(m.group(3), warnings, "train_outlier_pct")

            j = idx + 1
            while j < len(block_lines):
                sj = block_lines[j].strip()
                if sj.startswith("Test:") or "y-values distribution plot" in sj:
                    break
                m_out = re.match(r"-\s+(.+?)\s+\(([\d.]+)\s*SDs?\)", sj)
                if m_out:
                    result["train_outliers"].append({
                        "name": m_out.group(1).strip(),
                        "sd": safe_float(m_out.group(2), warnings, "train_outlier_sd"),
                    })
                j += 1

        m = re.search(r"Test:\s+(\d+) outliers out of (\d+) datapoints \(([\d.]+)%\)", s)
        if m:
            result["test_outlier_count"] = safe_int(m.group(1), warnings, "test_outlier_count")
            result["test_outlier_pct"] = safe_float(m.group(3), warnings, "test_outlier_pct")

            j = idx + 1
            while j < len(block_lines):
                sj = block_lines[j].strip()
                if "y-values distribution plot" in sj or sj.startswith("-------"):
                    break
                m_out = re.match(r"-\s+(.+?)\s+\(([\d.]+)\s*SDs?\)", sj)
                if m_out:
                    result["test_outliers"].append({
                        "name": m_out.group(1).strip(),
                        "sd": safe_float(m_out.group(2), warnings, "test_outlier_sd"),
                    })
                j += 1

    # ----------------------------------------------------------------
    # Quartile distribution and warning text
    # ----------------------------------------------------------------
    m = re.search(r"Q1:\s*(\d+),\s*Q2:\s*(\d+),\s*Q3:\s*(\d+),\s*Q4:\s*(\d+)", full_text)
    if m:
        result["quartile_counts"] = [
            safe_int(m.group(1)), safe_int(m.group(2)),
            safe_int(m.group(3)), safe_int(m.group(4)),
        ]

    for line in block_lines:
        s = line.strip()
        if "WARNING!" in s:
            result["warnings"].append(s)
            if "uniform" in s.lower() and result["y_distribution_warning"] is None:
                result["y_distribution_warning"] = s

    # ----------------------------------------------------------------
    # Feature-importance summary from "Influence on RMSE" lines
    # ----------------------------------------------------------------
    influence_mode = False
    for line in block_lines:
        s = line.strip()
        if s.startswith("Influence on RMSE") or s.startswith("Influence on MCC"):
            influence_mode = True
            continue
        if influence_mode:
            if s.startswith("-"):
                m = re.match(r"-\s+(.+?)\s*=\s*([\d.eE+\-]+)\s*\+\-\s*([\d.eE+\-]+)", s)
                if m:
                    result["feature_importance_summary"].append({
                        "descriptor": m.group(1).strip(),
                        "influence_rmse": safe_float(m.group(2), warnings, "feature_influence_rmse"),
                        "std": safe_float(m.group(3), warnings, "feature_influence_std"),
                    })
            else:
                if "Pearson heatmap" in s or "Outliers plot" in s or s.startswith("o"):
                    break

    return result


# Run the parser on both blocks
predict_warnings = []

if avail_predict:
    predict_blocks = split_into_blocks(predict_lines, NO_PFI_MARKER, PFI_MARKER)
    predict_no_pfi = parse_predict_block(predict_blocks["no_pfi"], predict_warnings)
    predict_pfi    = parse_predict_block(predict_blocks["pfi"],    predict_warnings)
    avail_pfi      = bool(predict_blocks["pfi"])
    avail_test_set = (
        predict_no_pfi["r2_test"] is not None or
        predict_no_pfi["mcc_test"] is not None
    )

    ml_model = predict_no_pfi["model"]

    if re.search(r"\d+x\s+\d+-fold CV\s*:\s+R2\s*=", "\n".join(predict_lines)):
        pred_type = "reg"
    elif re.search(r"\d+x\s+\d+-fold CV\s*:\s+Accur\.\s*=", "\n".join(predict_lines)) or re.search(r"\d+x\s+\d+-fold CV\s*:\s+MCC\s*=", "\n".join(predict_lines)):
        pred_type = "clas"
    else:
        pred_type = None
        predict_warnings.append("Could not detect pred_type from CV line format.")
else:
    predict_no_pfi = {k: None for k in [
        "cv_type", "points_descp_ratio", "r2_cv", "r2_test",
        "rmse_cv", "rmse_test", "mae_cv", "mae_test", "avg_sd_test",
        "y_min", "y_max", "y_range", "n_train", "n_test",
        "n_descriptors", "descriptors", "model", "target", "kfold",
        "cv_repeats", "train_outlier_count", "train_outlier_pct",
        "test_outlier_count", "test_outlier_pct", "quartile_counts",
        "mcc_cv", "mcc_test", "f1_cv", "f1_test", "accuracy_cv", "accuracy_test",
    ]}
    predict_no_pfi["train_outliers"] = []
    predict_no_pfi["test_outliers"] = []
    predict_no_pfi["feature_importance_summary"] = []
    predict_no_pfi["artifacts"] = {}
    predict_no_pfi["warnings"] = []

    predict_pfi = json.loads(json.dumps(predict_no_pfi))
    avail_pfi = False
    avail_test_set = False
    ml_model = None
    pred_type = None

print(f"Prediction type detected : {pred_type}")
print(f"ML model                 : {ml_model}")
print(f"PFI block present        : {avail_pfi}")
print(f"Test set metrics present : {avail_test_set}")
print()
print("--- No PFI model summary ---")
print(f"  CV/Test core metrics   : R2CV={predict_no_pfi['r2_cv']}, R2Test={predict_no_pfi['r2_test']}, RMSECV={predict_no_pfi['rmse_cv']}, RMSETest={predict_no_pfi['rmse_test']}")
print(f"  Clas metrics (if any)  : MCCCV={predict_no_pfi['mcc_cv']}, MCCTest={predict_no_pfi['mcc_test']}, F1CV={predict_no_pfi['f1_cv']}, F1Test={predict_no_pfi['f1_test']}, ACCCV={predict_no_pfi['accuracy_cv']}, ACCTest={predict_no_pfi['accuracy_test']}")
print(f"  Descriptors ({predict_no_pfi['n_descriptors']}): {predict_no_pfi['descriptors']}")
print(f"  Train outliers listed  : {len(predict_no_pfi['train_outliers'])}")
print()
print("--- PFI model summary ---")
print(f"  CV/Test core metrics   : R2CV={predict_pfi['r2_cv']}, R2Test={predict_pfi['r2_test']}, RMSECV={predict_pfi['rmse_cv']}, RMSETest={predict_pfi['rmse_test']}")
print(f"  Clas metrics (if any)  : MCCCV={predict_pfi['mcc_cv']}, MCCTest={predict_pfi['mcc_test']}, F1CV={predict_pfi['f1_cv']}, F1Test={predict_pfi['f1_test']}, ACCCV={predict_pfi['accuracy_cv']}, ACCTest={predict_pfi['accuracy_test']}")
print(f"  Descriptors ({predict_pfi['n_descriptors']}): {predict_pfi['descriptors']}")
print(f"  Train outliers listed  : {len(predict_pfi['train_outliers'])}")
if predict_warnings:
    print(f"\nPREDICT parser warnings: {predict_warnings}")

Prediction type detected : reg
ML model                 : NN
PFI block present        : True
Test set metrics present : True

--- No PFI model summary ---
  CV/Test core metrics   : R2CV=0.79, R2Test=1.0, RMSECV=0.53, RMSETest=0.54
  Clas metrics (if any)  : MCCCV=None, MCCTest=None, F1CV=None, F1Test=None, ACCCV=None, ACCTest=None
  Descriptors (6): ['C(CF3)3', 'CH2-n-Bu', 'G solv. in H2O', 'O1CC1_O_Dispersion', 'O1CC1_O_s proportion', 'PhePhe']
  Train outliers listed  : 0

--- PFI model summary ---
  CV/Test core metrics   : R2CV=0.71, R2Test=0.98, RMSECV=0.63, RMSETest=0.32
  Clas metrics (if any)  : MCCCV=None, MCCTest=None, F1CV=None, F1Test=None, ACCCV=None, ACCTest=None
  Descriptors (2): ['O1CC1_O_Dispersion', 'O1CC1_O_Pyramidaliz. volume']
  Train outliers listed  : 1


## Cell 10 Guide: Parse VERIFY_data.dat

The VERIFY module tests whether the ML model is better than three simple baselines:

- `y_mean`: a model that predicts the mean value for everything (trivial baseline),
- `y_shuffle`: a model trained on randomly scrambled target values (random baseline),
- `onehot`: a model trained on arbitrary one-hot labels (structure-free baseline).

If your model cannot beat these baselines, it is essentially memorizing noise.

VERIFY also tests whether the model can predict the held-out extremes of the dataset
(sorted cross-validation), which reveals whether the model extrapolates or just interpolates.

This cell extracts: how many tests were passed, failed, or unclear — and the RMSE
values from sorted cross-validation.

In [83]:
def parse_verify_block(block_lines, warnings):
    """
    Extract verification test results from one section (No PFI or PFI) of VERIFY_data.dat.
    """
    result = {
        "failed_tests"       : None,
        "unclear_tests"      : None,
        "passed_tests"       : None,
        "test_results"       : None,
        "test_outcomes"      : [],
        "flawed_mod_score"   : None,
        "sorted_cv_rmse"     : None,
        "sorted_cv_r2"       : None,
        "sorted_cv_mae"      : None,
        "sorted_cv_mcc"      : None,
        "cv_rmse_original"   : None,
        "threshold_15"       : None,
        "threshold_30"       : None,
        "warnings"           : [],
    }

    if not block_lines:
        return result

    full_text = "\n".join(block_lines)

    # Original RMSE/MCC and alert thresholds
    m = re.search(
        r"Original (?:RMSE|MCC) \(.*?\)\s+([\d.eE+\-]+)\s+\+\s+15%\s*&\s*30%\s*threshold\s*=\s*([\d.eE+\-]+)\s*&\s*([\d.eE+\-]+)",
        full_text,
    )
    if m:
        result["cv_rmse_original"] = safe_float(m.group(1), warnings, "cv_rmse_original")
        result["threshold_15"] = safe_float(m.group(2), warnings, "threshold_15")
        result["threshold_30"] = safe_float(m.group(3), warnings, "threshold_30")

    # Baseline test outcomes
    # Example: o y_mean: PASSED, RMSE = 1.2e+01, higher than thresholds
    test_pattern = re.compile(
        r"[ox\-]\s+(y_mean|y_shuffle|onehot):\s+(PASSED|UNCLEAR|FAILED)(?:,\s*(RMSE|MCC)\s*=\s*([\d.eE+\-]+))?(?:,\s*(.+))?"
    )
    matches = test_pattern.findall(full_text)

    if matches:
        outcomes = []
        for test_name, verdict, metric_name, metric_value, tail_text in matches:
            outcomes.append({
                "test": test_name,
                "verdict": verdict,
                "metric_name": metric_name if metric_name else None,
                "metric_value": safe_float(metric_value, warnings, f"verify_{test_name}_metric") if metric_value else None,
                "note": tail_text.strip() if tail_text else None,
            })
        result["test_outcomes"] = outcomes
        result["test_results"] = [f"{o['test']}: {o['verdict']}" for o in outcomes]
        result["passed_tests"] = sum(1 for o in outcomes if o["verdict"] == "PASSED")
        result["unclear_tests"] = sum(1 for o in outcomes if o["verdict"] == "UNCLEAR")
        result["failed_tests"] = sum(1 for o in outcomes if o["verdict"] == "FAILED")
        result["flawed_mod_score"] = (-1 * result["unclear_tests"]) + (-2 * result["failed_tests"])
    else:
        result["test_results"] = []
        result["test_outcomes"] = []
        result["passed_tests"] = 0
        result["unclear_tests"] = 0
        result["failed_tests"] = 0
        result["flawed_mod_score"] = 0

    # Sorted CV values
    m_rmse = re.search(r"-\s+Sorted \d+-fold CV\s*:.*?RMSE\s*=\s*(\[[^\]]+\])", full_text)
    if m_rmse:
        try:
            result["sorted_cv_rmse"] = json.loads(m_rmse.group(1))
        except Exception:
            warnings.append(f"Could not parse sorted CV RMSE list: {m_rmse.group(1)}")

    m_r2 = re.search(r"-\s+Sorted \d+-fold CV\s*:.*?R2\s*=\s*(\[[^\]]+\])", full_text)
    if m_r2:
        try:
            result["sorted_cv_r2"] = json.loads(m_r2.group(1))
        except Exception:
            warnings.append(f"Could not parse sorted CV R2 list: {m_r2.group(1)}")

    m_mae = re.search(r"-\s+Sorted \d+-fold CV\s*:.*?MAE\s*=\s*(\[[^\]]+\])", full_text)
    if m_mae:
        try:
            result["sorted_cv_mae"] = json.loads(m_mae.group(1))
        except Exception:
            warnings.append(f"Could not parse sorted CV MAE list: {m_mae.group(1)}")

    m_mcc = re.search(r"-\s+Sorted \d+-fold CV\s*:.*?MCC\s*=\s*(\[[^\]]+\])", full_text)
    if m_mcc:
        try:
            result["sorted_cv_mcc"] = json.loads(m_mcc.group(1))
        except Exception:
            warnings.append(f"Could not parse sorted CV MCC list: {m_mcc.group(1)}")

    # Warning lines printed by ROBERT in VERIFY
    for line in block_lines:
        s = line.strip()
        if "WARNING" in s or s.startswith("x "):
            result["warnings"].append(s)

    return result

In [84]:
# Run the parser on both VERIFY blocks
verify_warnings = []

empty_verify = {
    "failed_tests": None,
    "unclear_tests": None,
    "passed_tests": None,
    "test_results": None,
    "test_outcomes": [],
    "flawed_mod_score": None,
    "sorted_cv_rmse": None,
    "sorted_cv_r2": None,
    "sorted_cv_mae": None,
    "sorted_cv_mcc": None,
    "cv_rmse_original": None,
    "threshold_15": None,
    "threshold_30": None,
    "warnings": [],
}

if avail_verify:
    verify_blocks = split_into_blocks(verify_lines, NO_PFI_MARKER, PFI_MARKER)
    verify_no_pfi = parse_verify_block(verify_blocks["no_pfi"], verify_warnings)
    verify_pfi = parse_verify_block(verify_blocks["pfi"], verify_warnings)
else:
    verify_blocks = {"no_pfi": [], "pfi": []}
    verify_no_pfi = json.loads(json.dumps(empty_verify))
    verify_pfi = json.loads(json.dumps(empty_verify))

print("--- VERIFY summary ---")
print(
    f"No PFI: passed={verify_no_pfi['passed_tests']}, unclear={verify_no_pfi['unclear_tests']}, "
    f"failed={verify_no_pfi['failed_tests']}, score={verify_no_pfi['flawed_mod_score']}"
)
print(
    f"PFI   : passed={verify_pfi['passed_tests']}, unclear={verify_pfi['unclear_tests']}, "
    f"failed={verify_pfi['failed_tests']}, score={verify_pfi['flawed_mod_score']}"
)
if verify_warnings:
    print(f"VERIFY parser warnings: {verify_warnings}")

--- VERIFY summary ---
No PFI: passed=3, unclear=0, failed=0, score=0
PFI   : passed=3, unclear=0, failed=0, score=0


## Cell 12 Guide: Parse CURATE_data.dat and run_manifest.json

The CURATE module cleans and filters the input dataset before model training. It:
- removes duplicate compounds,
- removes descriptors (molecular features) that are nearly perfectly correlated with each other,
- applies recursive feature elimination (RFECV) if the dataset is large enough.

This cell extracts the initial dataset size, how many features were available before curation,
and how many were kept after filtering.

It also reads the `run_manifest.json` to capture the exact ROBERT command used.

In [85]:
def parse_curate_dat(lines, warnings):
    """
    Extract dataset and feature counts from CURATE_data.dat.
    """
    result = {
        "n_initial"          : None,
        "n_final"            : None,
        "n_dropped"          : None,
        "n_features_initial" : None,
        "n_features_final"   : None,
        "n_features_removed" : None,
        "ignored_descriptors": None,
        "curate_warnings"    : [],
    }

    if not lines:
        return result

    full_text = "\n".join(lines)

    m = re.search(r"Database .+? loaded successfully", full_text)
    if m:
        first_load_text = full_text[m.start():m.start()+400]

        m2 = re.search(r"(\d+) datapoints", first_load_text)
        if m2:
            result["n_initial"] = safe_int(m2.group(1), warnings, "n_initial")

        m2 = re.search(r"(\d+) accepted descriptors", first_load_text)
        if m2:
            result["n_features_initial"] = safe_int(m2.group(1), warnings, "n_features_initial")

        m2 = re.search(r"(\d+) ignored descriptors", first_load_text)
        if m2:
            result["ignored_descriptors"] = safe_int(m2.group(1), warnings, "ignored_descriptors")

    m = re.search(r"Total:\s+(\d+) descriptors removed due to high correlation", full_text)
    if m:
        result["n_features_removed"] = safe_int(m.group(1), warnings, "n_features_removed")

    if result["n_features_initial"] is not None:
        removed = result["n_features_removed"] or 0
        result["n_features_final"] = result["n_features_initial"] - removed

    if re.search(r"No datapoints were removed", full_text):
        result["n_final"] = result["n_initial"]
        result["n_dropped"] = 0
    else:
        dropped_matches = re.findall(r"(\d+) datapoints? (?:excluded|removed|discarded)", full_text)
        if dropped_matches:
            total_dropped = sum(safe_int(x) or 0 for x in dropped_matches)
            if result["n_initial"] is not None:
                result["n_final"] = result["n_initial"] - total_dropped
                result["n_dropped"] = total_dropped

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("x ") or "WARNING" in stripped:
            result["curate_warnings"].append(stripped)

    return result


def parse_generate_dat(lines, warnings):
    """
    Extract model-screening and feature-importance related summaries from GENERATE_data.dat.
    """
    result = {
        "model_screening_summary": [],
        "generate_warnings": [],
    }

    if not lines:
        return result

    for line in lines:
        s = line.strip()

        m = re.search(
            r"Best combined\s+(RMSE|MCC).*?for\s+([A-Za-z0-9_\-]+).*?:\s*([\d.eE+\-]+)",
            s,
        )
        if not m:
            m = re.search(
                r"Combined\s+(RMSE|MCC)\s+for\s+([A-Za-z0-9_\-]+).*?:\s*([\d.eE+\-]+)",
                s,
            )
        if m:
            result["model_screening_summary"].append({
                "metric": m.group(1),
                "model": m.group(2),
                "value": safe_float(m.group(3), warnings, "generate_model_screening_value"),
                "raw": s,
            })

        if s.startswith("x ") or "WARNING" in s:
            result["generate_warnings"].append(s)

    return result


def parse_robert_version_from_lines(*line_lists):
    for lines in line_lists:
        if not lines:
            continue
        for line in lines[:10]:
            m = re.search(r"ROBERT\s+v\s+([0-9][0-9A-Za-z._\-]+)", line)
            if m:
                return m.group(1)
    return None


def parse_command_from_lines(*line_lists):
    for lines in line_lists:
        if not lines:
            continue
        for line in lines:
            m = re.search(r"Command line used in ROBERT:\s*(.+)", line)
            if m:
                return m.group(1).strip()
    return None


curate_warnings = []
generate_warnings = []

if avail_curate:
    curate_info = parse_curate_dat(curate_lines, curate_warnings)
else:
    curate_info = {
        "n_initial": None, "n_final": None, "n_dropped": None,
        "n_features_initial": None, "n_features_final": None,
        "n_features_removed": None, "ignored_descriptors": None,
        "curate_warnings": [],
    }

if avail_generate:
    generate_info = parse_generate_dat(generate_lines, generate_warnings)
else:
    generate_info = {
        "model_screening_summary": [],
        "generate_warnings": [],
    }

# Manifest provenance
robert_command_used = None
dataset_csv_used = None
robert_version = None

if manifest_data:
    robert_command_used = manifest_data.get("robert_command")
    dataset_csv_used = manifest_data.get("dataset_csv")
    robert_version = manifest_data.get("robert_version")

# Fallback to headers in ROBERT output files when manifest does not include these
if robert_version is None:
    robert_version = parse_robert_version_from_lines(predict_lines, verify_lines, generate_lines, curate_lines)

if robert_command_used is None:
    command_line_text = parse_command_from_lines(predict_lines, verify_lines, generate_lines, curate_lines)
    if command_line_text:
        robert_command_used = command_line_text.split()

print("--- CURATE summary ---")
print(f"  Initial datapoints     : {curate_info['n_initial']}")
print(f"  Final datapoints       : {curate_info['n_final']}  (dropped: {curate_info['n_dropped']})")
print(f"  Initial features       : {curate_info['n_features_initial']}")
print(f"  Features after curation: {curate_info['n_features_final']}  (removed: {curate_info['n_features_removed']})")
print(f"  Ignored columns        : {curate_info['ignored_descriptors']}")
if curate_info["curate_warnings"]:
    print(f"  CURATE warnings        : {curate_info['curate_warnings']}")

print()
print("--- GENERATE summary ---")
print(f"  Model screening entries: {len(generate_info['model_screening_summary'])}")
if generate_info["model_screening_summary"]:
    print(f"  First entry            : {generate_info['model_screening_summary'][0]}")

print()
print("--- Provenance ---")
print(f"  ROBERT version: {robert_version}")
print(f"  Dataset CSV   : {dataset_csv_used}")
print(f"  Command       : {' '.join(robert_command_used) if isinstance(robert_command_used, list) and robert_command_used else robert_command_used}")

--- CURATE summary ---
  Initial datapoints     : 19
  Final datapoints       : 19  (dropped: 0)
  Initial features       : 40
  Features after curation: 16  (removed: 24)
  Ignored columns        : 1
  CURATE warnings        : ['x  The Pearson heatmap was not generated because the number of features and the y value (32) is higher than 30.']

--- GENERATE summary ---
  Model screening entries: 8
  First entry            : {'metric': 'RMSE', 'model': 'RF', 'value': 1.3, 'raw': 'o Best combined RMSE (target) found in BO for RF (no PFI filter): 1.3'}

--- Provenance ---
  ROBERT version: 2.1.0
  Dataset CSV   : /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/AQME-ROBERT_interpret_CO2.csv
  Command       : python -m robert --names SMILES --y MolLogP --csv_name /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/AQME-ROBERT_interpret_CO2.csv


## Cell 14 Guide: Assemble and Write run_context.json

This cell combines everything extracted above into a single structured file: `run_context.json`.

The file is written to the run folder alongside the archived ROBERT output folders.
It follows the V1 schema defined in `agent/run_context_schema.md`.

The JSON file is then printed in a human-readable form so you can visually check
that the extraction looks correct.

If any field shows `null`, that means either the relevant ROBERT output was missing,
or the parser could not find that value in the output file.

In [86]:
from datetime import datetime, timezone
import json
import shutil

from robert.report_utils import calc_score


def _score_components_from_data_score(data_score, suffix):
    components = {}
    suffix_tail = f"_{suffix}"
    for key, value in data_score.items():
        if not key.endswith(suffix_tail):
            continue
        if key.startswith("robert_score_"):
            continue
        components[key[: -len(suffix_tail)]] = value
    return components


all_warnings = []
for warning_list in [
    predict_warnings,
    verify_warnings,
    curate_warnings,
    generate_warnings,
    report_pdf_warnings,
]:
    if isinstance(warning_list, list):
        all_warnings.extend(warning_list)

score_no_pfi = None
score_pfi = None
score_no_pfi_components = None
score_pfi_components = None

if isinstance(predict_lines, list) and isinstance(verify_lines, list):
    try:
        dat_files = {"PREDICT": predict_lines, "VERIFY": verify_lines}
        data_score = {}

        data_score = calc_score(dat_files, "No PFI", pred_type, data_score)
        score_no_pfi = data_score.get("robert_score_No PFI")
        score_no_pfi_components = _score_components_from_data_score(data_score, "No PFI")

        if avail_pfi:
            data_score = calc_score(dat_files, "PFI", pred_type, data_score)
            score_pfi = data_score.get("robert_score_PFI")
            score_pfi_components = _score_components_from_data_score(data_score, "PFI")

    except Exception as exc:
        all_warnings.append(
            "Could not reconstruct ROBERT score from PREDICT/VERIFY data using ROBERT report logic: "
            f"{exc}"
        )
else:
    all_warnings.append(
        "Could not reconstruct ROBERT score because PREDICT_data.dat or VERIFY_data.dat was unavailable."
    )


run_context = {
    "schema_version": "1.2",
    "extracted_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "results_dir": str(RUN_FOLDER.resolve()),
    "pred_type": pred_type,
    "ml_model": ml_model,
    "robert_version": robert_version,
    "dataset_csv": dataset_csv_used,
    "robert_command": robert_command_used,
    "report_assets": {"pdf_files": report_pdfs},
    "available": {
        "predict": avail_predict,
        "verify": avail_verify,
        "curate": avail_curate,
        "generate": avail_generate,
        "test_set": avail_test_set,
        "pfi": avail_pfi,
        "report_pdf": len(report_pdfs) > 0,
    },
    "predict": {
        "no_pfi": predict_no_pfi,
        "pfi": predict_pfi,
    },
    "verify": {
        "no_pfi": verify_no_pfi,
        "pfi": verify_pfi,
    },
    "curate": {
        "n_initial": curate_info["n_initial"],
        "n_final": curate_info["n_final"],
        "n_dropped": curate_info["n_dropped"],
        "n_features_initial": curate_info["n_features_initial"],
        "n_features_final": curate_info["n_features_final"],
        "n_features_removed": curate_info["n_features_removed"],
        "ignored_descriptors": curate_info["ignored_descriptors"],
        "warnings": curate_info["curate_warnings"],
    },
    "generate": {
        "model_screening_summary": generate_info["model_screening_summary"],
        "warnings": generate_info["generate_warnings"],
    },
    "score": {
        "no_pfi": score_no_pfi,
        "pfi": score_pfi,
        "no_pfi_components": score_no_pfi_components,
        "pfi_components": score_pfi_components,
    },
    "parser_warnings": all_warnings,
}

run_context_path = RUN_FOLDER / "run_context.json"
with open(run_context_path, "w", encoding="utf-8") as f:
    json.dump(run_context, f, indent=2, ensure_ascii=False)

llm_evidence = {
    "schema_version": "1.0",
    "built_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "source_run_dir": str(RUN_FOLDER.resolve()),
    "source_of_truth": "ROBERT outputs",
    "policy": {
        "pdf_for_view_only": True,
        "llm_must_use_extracted_evidence_only": True,
        "no_competing_score_system": True,
    },
    "provenance": {
        "robert_version": robert_version,
        "robert_command": robert_command_used,
        "dataset_csv": dataset_csv_used,
    },
    "report_assets": {"pdf_files": report_pdfs},
    "score": run_context.get("score", {}),
    "predict": run_context.get("predict", {}),
    "verify": run_context.get("verify", {}),
    "curate": run_context.get("curate", {}),
    "generate": run_context.get("generate", {}),
    "parser_warnings": run_context.get("parser_warnings", []),
}

llm_evidence_path = RUN_FOLDER / "llm_evidence.json"
with open(llm_evidence_path, "w", encoding="utf-8") as f:
    json.dump(llm_evidence, f, indent=2, ensure_ascii=False)

bundle_dir = RUN_FOLDER / "llm_run_bundle"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(run_context_path, bundle_dir / "run_context.json")
shutil.copy2(llm_evidence_path, bundle_dir / "llm_evidence.json")

bundle_report_assets = []
report_assets_src = RUN_FOLDER / "report_assets"
if report_assets_src.exists():
    report_assets_dst = bundle_dir / "report_assets"
    shutil.copytree(report_assets_src, report_assets_dst)
    for item in report_pdfs:
        bundle_report_assets.append(
            {
                "name": item.get("name"),
                "path": str((report_assets_dst / item.get("name", "")).resolve()),
                "source_path": item.get("path"),
            }
        )

bundle_index = {
    "schema_version": "1.0",
    "bundle_created_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "bundle_dir": str(bundle_dir.resolve()),
    "files": {
        "run_context": str((bundle_dir / "run_context.json").resolve()),
        "llm_evidence": str((bundle_dir / "llm_evidence.json").resolve()),
        "report_assets": bundle_report_assets,
    },
    "routing": {
        "ui_pdf_view_source": "report_assets",
        "llm_prompt_source": "llm_evidence.json",
        "note": "LLM should use extracted evidence only, not PDF text.",
    },
}

bundle_index_path = bundle_dir / "bundle_index.json"
with open(bundle_index_path, "w", encoding="utf-8") as f:
    json.dump(bundle_index, f, indent=2, ensure_ascii=False)

print("run_context.json written to:")
print(f"  {run_context_path}")
print("llm_evidence.json written to:")
print(f"  {llm_evidence_path}")
print("LLM bundle directory:")
print(f"  {bundle_dir}")
print("LLM bundle index:")
print(f"  {bundle_index_path}")
print()
print("run_context score block:")
print(json.dumps(run_context.get("score", {}), indent=2, ensure_ascii=False))
print()
print("report assets:")
print(json.dumps(run_context.get("report_assets", {}), indent=2, ensure_ascii=False))
print()
print(f"parser warnings (count): {len(run_context.get('parser_warnings', []))}")

run_context.json written to:
  /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260521_062044__AQME-ROBERT_interpret_CO2/run_context.json
llm_evidence.json written to:
  /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260521_062044__AQME-ROBERT_interpret_CO2/llm_evidence.json
LLM bundle directory:
  /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260521_062044__AQME-ROBERT_interpret_CO2/llm_run_bundle
LLM bundle index:
  /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260521_062044__AQME-ROBERT_interpret_CO2/llm_run_bundle/bundle_index.json

run_context score block:
{
  "no_pfi": 6,
  "pfi": 7,
  "no_pfi_components": {
    "rmse_score": 0,
    "cv_type": "10x 5-fold CV",
    "points_descp_ratio": "15:6",
    "rmse_cv": 0.53,
    "r2_cv": 0.79,
    "rmse_test": 0.54,
    "r2_test": 1.0,
    "y_range": 4.2,
    "scaled_rmse_cv": 12.62,
    "scaled_rmse_test": 12.86,
    "cv_score_rmse": 1,
    "test_score_rmse": 1,
    "cv_penalty_r2": 0,
  

## Usage Notes

### How to use this notebook on a new run

1. Run `agent/robert_run_wrapper.ipynb` to generate and archive a ROBERT run.
2. Open this notebook and update `RUN_FOLDER` in Cell 3 to point at the new archive,
   or leave it as `None` to auto-select the latest run.
3. Run all cells from top to bottom.
4. Check the printed summary from each section for unexpected `None` values.
5. The notebook writes two backend artifacts in the run folder:
   - `run_context.json` (full extraction record)
   - `llm_evidence.json` (LLM-facing evidence payload)

### PDF and LLM behavior

- Report PDFs are copied into `report_assets/` inside the run archive when found.
- The UI can show those PDFs directly to users for viewing.
- The LLM should use `llm_evidence.json` (not PDF text) as its evidence source.

### Interpreting null values

- `null` for any metric field means that line was not found in the `.dat` file.
- `null` for score values means a ROBERT score string was not present in available `.dat` files.
- Check `parser_warnings` in the JSON for extraction gaps and parse issues.

## Cell 17 Guide: Build dataset_profile.json for the bot

This optional extraction cell creates one additional artifact: `dataset_profile.json`.

The profile summarizes the original input dataset and CURATE evidence in a compact,
machine-readable format for the companion bot. It avoids storing raw matrices or full
CSV content.

The profile is written into the same run folder as `run_context.json` and can be loaded
by the UI as extra context for chemist-facing explanations.

In [87]:
from profile_dataset import profile_dataset


dataset_profile = profile_dataset(RUN_FOLDER, write_json=True, include_smiles_summary=False)

dataset_profile_path = RUN_FOLDER / "dataset_profile.json"
print("dataset_profile.json written to:")
print(f"  {dataset_profile_path}")
print()
print("dataset_profile summary:")
print(f"  rows: {dataset_profile.get('row_count')}")
print(f"  columns: {dataset_profile.get('column_count')}")
print(f"  candidate descriptors: {dataset_profile.get('candidate_descriptor_count')}")
print(f"  constant columns: {dataset_profile.get('constant_columns')}")
print(f"  top descriptor-target correlations: {dataset_profile.get('top_descriptor_target_correlations')[:3]}")

dataset_profile.json written to:
  /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260521_062044__AQME-ROBERT_interpret_CO2/dataset_profile.json

dataset_profile summary:
  rows: 19
  columns: 42
  candidate descriptors: 40
  constant columns: ['O1CC1_O_d proportion', 'Total charge']
  top descriptor-target correlations: [{'descriptor': 'O1CC1_O_Dispersion', 'r2_with_target': 0.762792}, {'descriptor': 'Second EA', 'r2_with_target': 0.432318}, {'descriptor': 'O1CC1_O_Pyramidaliz. volume', 'r2_with_target': 0.405944}]
